In [1]:
import os
DATA_DIR = "/kaggle/input/datasets/nhttrangminh/deepfashion"
print(os.listdir(DATA_DIR))

['processed_for_embedding.csv', 'merged_dataset_clean.csv', 'merged_dataset_clean_with_text.csv', 'img']


In [2]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/nhttrangminh/deepfashion/processed_for_embedding.csv")
df.head()

,image_path,category,split,category_name,text,mapped_text,clip_text,combined_text
0,img/Sheer_Pleated-Front_Blouse/img_00000001.jpg,3,train,Sheer_Pleated-Front_Blouse,Sheer Pleated Front Blouse,Blouse,a photo of a Blouse,a fashion item. category: Blouse. product: She...
1,img/Sheer_Pleated-Front_Blouse/img_00000002.jpg,3,train,Sheer_Pleated-Front_Blouse,Sheer Pleated Front Blouse,Blouse,a photo of a Blouse,a fashion item. category: Blouse. product: She...
2,img/Sheer_Pleated-Front_Blouse/img_00000003.jpg,3,val,Sheer_Pleated-Front_Blouse,Sheer Pleated Front Blouse,Blouse,a photo of a Blouse,a fashion item. category: Blouse. product: She...
3,img/Sheer_Pleated-Front_Blouse/img_00000004.jpg,3,train,Sheer_Pleated-Front_Blouse,Sheer Pleated Front Blouse,Blouse,a photo of a Blouse,a fashion item. category: Blouse. product: She...
4,img/Sheer_Pleated-Front_Blouse/img_00000005.jpg,3,test,Sheer_Pleated-Front_Blouse,Sheer Pleated Front Blouse,Blouse,a photo of a Blouse,a fashion item. category: Blouse. product: She...


In [3]:
!pip install git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-lm31bcon
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-lm31bcon
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369547 sha256=d073c48437eb10515649609a13a37c4f8f19b883fc060c37ee32ad0a532796bd
  Stored in directory: /tmp/pip-ephem-wheel-cache-xuxjmsmm/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [ ]:
import torch
import clip
import pandas as pd
from tqdm import tqdm


device = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 256 

model, preprocess = clip.load("ViT-B/32", device=device)

df = pd.read_csv("/kaggle/input/datasets/nhttrangminh/deepfashion/processed_for_embedding.csv")

texts = df["combined_text"].tolist()

all_embeddings = []

for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch_texts = texts[i:i+BATCH_SIZE]

    text_tokens = clip.tokenize(batch_texts).to(device)

    with torch.no_grad():
        text_features = model.encode_text(text_tokens)

    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    all_embeddings.append(text_features.cpu())

text_embeddings = torch.cat(all_embeddings, dim=0)

print("Done:", text_embeddings.shape)

100%|████████████████████████████████████████| 338M/338M [00:02<00:00, 137MiB/s]
100%|██████████| 1130/1130 [03:19<00:00,  5.66it/s]


Done: torch.Size([289147, 512])


In [ ]:
import numpy as np

np.save("/kaggle/working/text_embeddings.npy", text_embeddings.numpy())

In [ ]:
import numpy as np
txt = np.load("text_embeddings.npy")
sample = txt[:2000]
unique_count = len(np.unique(sample.round(4), axis=0))
print(unique_count)

texts = df["combined_text"].tolist()

print("Total texts:", len(texts))
print("Unique texts:", len(set(texts)))

for i in range(10):
    print(texts[i])

17
Total texts: 289147
Unique texts: 5595
a fashion item. category: Blouse. product: Sheer Pleated Front Blouse. description: a photo of a Blouse
a fashion item. category: Blouse. product: Sheer Pleated Front Blouse. description: a photo of a Blouse
a fashion item. category: Blouse. product: Sheer Pleated Front Blouse. description: a photo of a Blouse
a fashion item. category: Blouse. product: Sheer Pleated Front Blouse. description: a photo of a Blouse
a fashion item. category: Blouse. product: Sheer Pleated Front Blouse. description: a photo of a Blouse
a fashion item. category: Blouse. product: Sheer Pleated Front Blouse. description: a photo of a Blouse
a fashion item. category: Blouse. product: Sheer Pleated Front Blouse. description: a photo of a Blouse
a fashion item. category: Blouse. product: Sheer Pleated Front Blouse. description: a photo of a Blouse
a fashion item. category: Blouse. product: Sheer Pleated Front Blouse. description: a photo of a Blouse
a fashion item. catego

In [ ]:
for i in range(0, len(texts), 256):
    batch = texts[i:i+256]
    
    print("Batch sample:")
    print(batch[:3])   
    
    tokens = clip.tokenize(batch).to(device)
    features = model.encode_text(tokens)
    
    break

Batch sample:
['a fashion item. category: Blouse. product: Sheer Pleated Front Blouse. description: a photo of a Blouse', 'a fashion item. category: Blouse. product: Sheer Pleated Front Blouse. description: a photo of a Blouse', 'a fashion item. category: Blouse. product: Sheer Pleated Front Blouse. description: a photo of a Blouse']


In [14]:
tokens = clip.tokenize(batch)

print(tokens[0][:10])
print(tokens[1][:10])
print(tokens[2][:10])

tensor([49406,   320,  2444,  8532,   269,  9432,   281, 23525,   269,  4306],
       dtype=torch.int32)
tensor([49406,   320,  2444,  8532,   269,  9432,   281, 23525,   269,  4306],
       dtype=torch.int32)
tensor([49406,   320,  2444,  8532,   269,  9432,   281, 23525,   269,  4306],
       dtype=torch.int32)


In [16]:
unique_count = len(np.unique(txt.round(4), axis=0))
print(unique_count)

5597


In [17]:
import random

idx = random.sample(range(len(txt)), 2000)
sample = txt[idx]

unique_count = len(np.unique(sample.round(4), axis=0))
print(unique_count)

1608


In [18]:
idx = np.random.choice(len(txt), 2000, replace=False)
sample = txt[idx]

unique_count = len(np.unique(sample.round(4), axis=0))
print(unique_count)

1631
